# Making chloropleth maps in Altair

Here's a quick example of how to make a chloropleth map in Altair.  In this example, we'll work with a fairly large data set of baby names in France from 1900-2019, broken down by department.

To work with geographical data, we'll use the `geopandas`, which loads `pandas` dataframes, but with support for geographical outlines in the `geojson` format.  You can use these dataframes just as you would a regular `pandas` dataframe, but they will include that extra geographical outline data.

To get started, we'll need to import our libraries.

In [1]:
import altair as alt
import pandas as pd
import geopandas as gpd # Requires geopandas -- e.g.: conda install -c conda-forge geopandas
_ = alt.data_transformers.enable('json') # Let Altair/Vega-Lite work with large data sets

# Reading our names data

Now, let's read in our dataset.  The exported data is in CSV format, but with a `;` separator instead of commas.  The INSEE data collapses rare names or where department-level information has been elided (presumably to protect individuals with uncommon names or who were one of the only ones born with that name in a given year).  We'll strip those out.

In [2]:
names = pd.read_csv("dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names.drop(names[names.dpt == 'XX'].index, inplace=True)

names.sample(5)

,sexe,preusuel,annais,dpt,nombre
3116575,2,MARIE-ROSE,1922,38,3
258814,1,BRYAN,1997,80,21
3100704,2,MARIE-LINE,1969,17,3
564295,1,FRANÇOIS,1968,38,82
2990171,2,MAGALI,1986,60,5


# Loading map data

Next, let's load some map data of regions in France using `geopandas`.  These map data come from the [INSEE] and [IGN] and were processed into the `geojson` format we'll need to work with by [Grégoire David].  Here's the [github] repository.

In this example, we'll work with the simplified departments tiles for the Hexagon, but that repository contains higher-resolution versions, the DOM-TOM, and more.

[Grégoire David]: https://gregoiredavid.fr
[INSEE]: http://www.insee.fr/fr/methodes/nomenclatures/cog/telechargement.asp
[IGN]: https://geoservices.ign.fr/adminexpress
[github]: https://github.com/gregoiredavid/france-geojson/

In [3]:
depts = gpd.read_file('departements-version-simplifiee.geojson')

depts.sample(5)

,code,nom,geometry
10,11,Aude,"POLYGON ((1.68842 43.27355, 1.70839 43.29127, ..."
83,83,Var,"MULTIPOLYGON (((6.4348 43.01554, 6.4552 43.026..."
57,57,Moselle,"POLYGON ((5.8934 49.49691, 5.93994 49.50097, 5..."
7,08,Ardennes,"POLYGON ((4.23316 49.95775, 4.3081 49.96952, 4..."
73,73,Savoie,"POLYGON ((6.80252 45.77837, 6.80842 45.72515, ..."


Notice how `depts` is a geopandas dataframe.  We'll use it just as a regular `pandas` dataframe, but it includes the geometry info we need to be able to draw those regions when we pass them into Altair.  We just need to make sure that when we work with our data, we keep them in a geopandas dataframe and not a plain dataframe if we want to draw the departments.

In the next cell, notice how we do a right-merge to bring in department data into names.  We do this as a merge on `depts` because we need a geopandas dataframe.  Remember, `depts` is a geopandas dataframe, while `names` is a regular dataframe.  If we did a left merge on `names`, we'd end up with a regular pandas dataframe. After this merge, both `names` and `depts` will be geopandas dataframes.

**Hint:** Be careful when you do your data joins here.  It's easy to accidentally merge the wrong way to accidentally create a _much bigger_ dataset.

In [4]:
# Keep a reference around to the plain pandas dataframe, without geometry data, just in case
just_names = names

names = depts.merge(names, how='right', left_on='code', right_on='dpt')

names.sample(5)

,code,nom,geometry,sexe,preusuel,annais,dpt,nombre
2080788,68,Haut-Rhin,"POLYGON ((7.19828 48.31047, 7.24173 48.30243, ...",2,CHARLINE,1989,68,11
3202683,33,Gironde,"POLYGON ((-0.7188 45.32742, -0.64431 45.32205,...",2,MORGANE,2016,33,11
1642955,31,Haute-Garonne,"POLYGON ((0.95398 43.78737, 0.9778 43.78644, 0...",1,XAVIER,1982,31,49
997156,53,Mayenne,"POLYGON ((-1.07016 48.50849, -1.06055 48.51534...",1,LOÏS,2013,53,3
1354744,75,Paris,"POLYGON ((2.41634 48.84924, 2.46226 48.84254, ...",1,RAPHAËL,1939,75,7


# Show a name over all years

Now we'll choose a name to show across all years.  To that, we'll group all of the names in a department together (squashing the years together) and use the sum.

In [8]:
grouped = names.groupby(['dpt', 'preusuel', 'sexe'], as_index=False).sum()
grouped = depts.merge(grouped, how='right', left_on='code', right_on='dpt') # Add geometry data back in
grouped

TypeError: agg function failed [how->sum,dtype->geometry]

Now let's pick a name and check out how it's distribution over the last 120 years across Metropolitan France.  In this example, I choose the name “Lucien,” which I rather like for some reason.

In [6]:
name = 'LUCIEN'
subset = grouped[grouped.preusuel == name]
alt.Chart(subset).mark_geoshape(stroke='white').encode(
    tooltip=['nom', 'code', 'nombre'],
    color='nombre',
).properties(width=800, height=600)

NameError: name 'grouped' is not defined